# Chapter 1 · Decisions before algorithms

This notebook accompanies *From Problems to Working Solutions*, Chapter 1. The clinic, rates and losses below are **teaching assumptions, not measured patient data**. Run the cells from top to bottom. No paid service or internet connection is required once the packages are installed.

## 1. Begin with a question

A registration queue seems long. Does that mean the facility should buy a forecasting system? Not necessarily. Begin by calculating what a simple queueing model actually predicts, and then ask which decision the calculation might inform. The model assumes Poisson arrivals, exponential service, a common queue and stable operation.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

lam, mu = 6.0, 8.0  # patients/hour, invented values
assert lam < mu
W = 1/(mu-lam)
Wq = lam/(mu*(mu-lam))
print(f"Time including service: {W*60:.1f} minutes")
print(f"Waiting before service: {Wq*60:.1f} minutes")
assert np.isclose(W*60, 30.0)
assert np.isclose(Wq*60, 22.5)

## 2. Compare one desk with two

For two or more servers, use the Erlang-C formula. Changing the number of servers changes the waiting-time calculation, but it also has staffing and downstream consequences that this model does not capture.

In [ ]:
def mmc(arrival, service, servers):
    if arrival >= servers*service or arrival < 0 or service <= 0 or servers < 1:
        raise ValueError("Model requires 0 <= arrival < c*service")
    a = arrival/service
    rho = a/servers
    p0 = 1/(sum(a**k/math.factorial(k) for k in range(servers)) +
            a**servers/(math.factorial(servers)*(1-rho)))
    pwait = a**servers*p0/(math.factorial(servers)*(1-rho))
    wq = pwait/(servers*service-arrival)
    return wq*60, (wq+1/service)*60, rho

scenarios = [("Current", 8, 1), ("Faster desk", 9, 1), ("Two desks", 8, 2)]
for name, rate, c in scenarios:
    print(name, tuple(round(v,3) for v in mmc(6,rate,c)))
assert np.isclose(mmc(6,8,2)[1], 8.7272727, atol=1e-6)

## 3. Why the average day can mislead

Here the model predicts rapidly rising mean time as arrivals approach the service rate. A real clinic with peaks and interruptions might depart from this curve. The figure is an analytical illustration, not a fitted prediction.

In [ ]:
rates=np.linspace(0.2,7.4,200)
w1=[mmc(rate,8,1)[1] for rate in rates]
w2=[mmc(rate,8,2)[1] for rate in rates]
plt.figure(figsize=(8,4))
plt.plot(rates,w1,label="One desk")
plt.plot(rates,w2,label="Two desks")
plt.xlabel("Assumed arrivals per hour")
plt.ylabel("Modelled time in registration (minutes)")
plt.title("A stable queue can become sensitive near full capacity")
plt.legend();plt.show()

## 4. A diagnostic test can be informative but still too costly

The two possible causes are registration (R) and laboratory (L). The rows are two feasible actions, and the columns are the possible causes. This loss table is invented and uses common hypothetical units.

In [ ]:
p = np.array([.6,.4])
loss = np.array([[2.,12.],[9.,3.]])
risk_each = loss @ p
risk_without = risk_each.min()
risk_perfect = p @ loss.min(axis=0)
evpi = risk_without-risk_perfect
print("Expected loss for each action:", risk_each)
print("Current best:",risk_without,"Perfect information:",risk_perfect,"EVPI:",evpi)
assert np.isclose(evpi,3.6)

## 5. What an imperfect test changes

Assume sensitivity 0.8 and specificity 0.75. Calculate the joint state/report probabilities and choose the best action separately for a positive and a negative report.

In [ ]:
signal = np.array([[.8,.25],[.2,.75]])  # rows: positive/negative; cols: R/L
joint = signal*p[None,:]
posterior = joint/joint.sum(axis=1,keepdims=True)
risks_by_report = loss@joint.T
risk_test = risks_by_report.min(axis=0).sum()
evsi = risk_without-risk_test
print("Posterior probability of R after positive/negative:",posterior[:,0])
print("Expected loss using test:",risk_test,"EVSI:",evsi)
assert np.isclose(risk_test,4.14) and np.isclose(evsi,1.86)
assert 0 <= evsi <= evpi

## 6. Find the point at which the action changes

The action improves registration when the assumed probability of a registration bottleneck exceeds 9/16. Notice that improving the accuracy of a probability estimate matters most near a decision boundary.

In [ ]:
grid=np.linspace(0,1,250)
r_registration=12-10*grid
r_laboratory=3+6*grid
plt.figure(figsize=(8,4))
plt.plot(grid,r_registration,label="Improve registration")
plt.plot(grid,r_laboratory,label="Improve laboratory workflow")
plt.axvline(9/16,ls="--",color="gray")
plt.xlabel("Assumed probability of registration bottleneck")
plt.ylabel("Expected loss (illustrative units)")
plt.legend();plt.show()
assert np.isclose(12-10*(9/16),3+6*(9/16))

## 7. Change the decision, not just the estimate

Try another prior probability for state R and another diagnostic-study cost. Report (a) the preferred action before the test, (b) the value of the test before cost, and (c) whether it is worth collecting the measurement in one decision period. Finally, name one observation you would need before making a real clinic recommendation.

In [ ]:
cost_of_test=2.0  # invented loss units, same decision period
print("Net one-period gain from diagnostic:",round(evsi-cost_of_test,2))
print("Would collect test on one-period criterion:",bool(evsi>cost_of_test))